# Train a model for Fink

This notebook is a template: copy it, then change the cells marked **✏️**.

You bring two things:

1. **your preprocessing**, in `preprocessing/preprocessing.py`: it turns one alert into the features of your model (section 1).
2. **your model**, trained below on the output of this preprocessing.

Everything is logged to MLflow for you: parameters, metrics, training data, model and preprocessing. The model is refused if it would not work in production (wrong features, preprocessing too slow...).

## 0. Connect to MLflow ✏️

In [1]:
import os
from getpass import getpass

os.environ.setdefault("MLFLOW_TRACKING_URI", "https://mlflow.fink-broker.org")
if "MLFLOW_TRACKING_USERNAME" not in os.environ:
    os.environ["MLFLOW_TRACKING_USERNAME"] = input("MLflow username: ")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass("MLflow password: ")

# Name of your model: lowercase letters, digits and dashes.
# It is also the name of the MLflow experiment: every run of this notebook goes there.
MODEL_NAME = "ztf-sso-flat"

## 1. Your preprocessing ✏️

Your preprocessing is in **`preprocessing/preprocessing.py`**: open and edit it there. It stays in its own file because this file is what runs in production, on every alert of the stream. This notebook always uses its last saved version.

- `FEATURE_NAMES`: the names of your features, in order;
- `pre_processing(alert)`: takes one alert (a `dict`) and returns one float per name.

Below: its code, then its result on one of your alerts.

In [2]:
import inspect

import fink_model
from fink_model.alerts import load_alerts

ALERTS = "/home/farid/Desktop/fink/fink-client/ftransfer_ztf_2026-10-06_534834"

prep = fink_model.preprocessing()
print("FEATURE_NAMES =", prep.FEATURE_NAMES, "\n")
print(inspect.getsource(prep.pre_processing))

FEATURE_NAMES = ['magpsf', 'sigmapsf', 'fid', 'nalerthist', 'mag_rate', 'sigma_rate', 'delta_time', 'days_since_first', 'from_upper', 'amplitude_g', 'amplitude_r', 'n_bands_with_lc'] 

def pre_processing(alert):
    """Return the feature vector of one alert, in the order of FEATURE_NAMES."""
    jd = _to_float(alert.get("jd"))
    amplitude_g = _amplitude(alert, "lc_features_g")
    amplitude_r = _amplitude(alert, "lc_features_r")

    return [_to_float(alert.get(name)) for name in DIRECT] + [
        max(0.0, jd - _to_float(alert.get("jd_first_real_det"), jd)),
        1.0 if alert.get("from_upper") is True else 0.0,
        amplitude_g or 0.0,
        amplitude_r or 0.0,
        float((amplitude_g is not None) + (amplitude_r is not None)),
    ]



In [3]:
alert = next(load_alerts(ALERTS))  # one alert, exactly as received in production
print(alert["objectId"], "->", dict(zip(prep.FEATURE_NAMES, prep.pre_processing(alert))))

ZTF26absuzhq -> {'magpsf': 0.0, 'sigmapsf': 0.0, 'fid': 0.0, 'nalerthist': 37.0, 'mag_rate': 0.019595636948187185, 'sigma_rate': 0.16640598470353687, 'delta_time': 0.9673727001063526, 'days_since_first': 0.0, 'from_upper': 0.0, 'amplitude_g': 0.1919110393078487, 'amplitude_r': 0.15071424529647537, 'n_bands_with_lc': 2.0}


## 2. Compute the features of your alerts ✏️

`load_features()` reads your alerts (a folder or a file: `.parquet`, `.avro`, `.jsonl`) and runs your `pre_processing()` on each of them, exactly as in production. It stops if the preprocessing fails on an alert or is too slow for the stream.

- `X`: one row of features per alert, in the order of `FEATURE_NAMES`;
- `info`: `objectId`, `candid`, and the `columns` you ask for, to build your labels.

Use `limit=5000` to try quickly on a part of the alerts.

In [4]:
X, info = fink_model.load_features(ALERTS, columns=["roid"], limit=None)
info.head()

39022 alerts, 12 features, 0.007 ms per alert


,objectId,candid,candidate.classtar
0,ZTF26absuzhq,3560140304115015058,0.982
1,ZTF26absuzhq,3560192994115015021,0.948
2,ZTF26abvdlgm,3560149125615015042,0.982
3,ZTF26abuxdqd,3560118260615015042,0.983
4,ZTF26abuxdqd,3560169810615015022,0.984


## 3. Your labels ✏️

One label per row of `X`. Build them from `info`, for example from a column of your files (`columns=["tnsclass"]`), or from your own list joined on `objectId`:

```python
labels = pd.read_csv("my_labels.csv")            # objectId,label
y = info.merge(labels, on="objectId", how="left")["label"]
```

**Example only**: here the label is "Solar System object known by the MPC" (`roid` == 3, about 8% of the alerts). Replace it with yours.

In [5]:
y = (info["roid"] == 3).astype(int)  # 3 = Solar System object known by the MPC
y.value_counts()

candidate.classtar
1    25341
0    13681
Name: count, dtype: int64

## 4. Your model ✏️

Any model with a `predict()` method: scikit-learn, XGBoost, LightGBM...

In production the model receives the output of `pre_processing()` and nothing else. If you transform `X` (scaling, feature selection...), do it **inside the model**, with a scikit-learn `Pipeline`:

```python
model = make_pipeline(StandardScaler(), LogisticRegression())   # not: X = StandardScaler().fit_transform(X)
```

The split keeps all the alerts of one object on the same side: with a random split, the test score is too optimistic.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train, test = next(splitter.split(X, y, groups=info["objectId"]))
X_train, X_test, y_train, y_test = X[train], X[test], y.iloc[train], y.iloc[test]

model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)

## 5. Train and log to MLflow

Inside `start_run()`, the parameters and the training metrics of your model are logged automatically. Log your own test metrics with `mlflow.log_metrics()`.

`log_model()` checks the model against your preprocessing, then uploads both to the run. Nothing is registered: you choose later which run to deploy.

In [ ]:
import mlflow
from sklearn.metrics import accuracy_score, f1_score

with fink_model.start_run(MODEL_NAME):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mlflow.log_metrics({
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    })

2026/10/07 14:05:27 INFO mlflow.tracking.fluent: Experiment with name 'ztf-sso-flat' does not exist. Creating a new experiment.
/home/farid/Desktop/fink/model_template/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/home/farid/Desktop/fink/model_template/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
Successfully registered model 'ztf-sso-flat'.
2026/10/07 14:05:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ztf-sso-flat, version 1
Created version '1' of model 'ztf-sso-flat

Uploaded ztf-sso-flat version 1 (run 0772df0fcf604a748adb524a4db72d70).
🏃 View run ztf-sso-flat at: https://mlflow.fink-broker.org/#/experiments/8/runs/0772df0fcf604a748adb524a4db72d70
🧪 View experiment at: https://mlflow.fink-broker.org/#/experiments/8


## 6. Deploy

Compare your runs in the MLflow interface (experiment `MODEL_NAME`). To deploy one, open it and click **Register model**, with `MODEL_NAME` as the name. Then, to build the two containers (preprocessing and model) and run your model on Fink alerts, follow the [Fink AI documentation](https://doc.ztf.fink-broker.org/services/fink_ai/).